# Twitter Sentiment Analysis

## Installing the Libraries

In [1]:
import pandas as pd ## to handle the dataframes
import numpy as np
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer ## vectorization of words
from sklearn.model_selection import train_test_split ## test and train ko alag klarne ke liye
from sklearn.naive_bayes import BernoulliNB ### what is this?
from sklearn.linear_model import LogisticRegression ## model ha bhai
from sklearn.svm import LinearSVC  ## model ha but why this?
from sklearn.metrics import accuracy_score, classification_report ## Accuracy ke liye

Things I dont know are
1. re
2. BernourlliNB

### downloading the stopwords

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Error loading stopwords: [ASN1: NOT_ENOUGH_DATA] not
[nltk_data]     enough data (_ssl.c:4040)


False

## ## Data Processing

In [3]:
df = pd.read_csv('training.1600000.processed.noemoticon.csv', encoding= 'ISO-8859-1')
df.head(10)

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew
5,0,1467811592,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,mybirch,Need a hug
6,0,1467811594,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,coZZ,@LOLTrish hey long time no see! Yes.. Rains a...
7,0,1467811795,Mon Apr 06 22:20:05 PDT 2009,NO_QUERY,2Hood4Hollywood,@Tatiana_K nope they didn't have it
8,0,1467812025,Mon Apr 06 22:20:09 PDT 2009,NO_QUERY,mimismo,@twittera que me muera ?
9,0,1467812416,Mon Apr 06 22:20:16 PDT 2009,NO_QUERY,erinx3leannexo,spring break in plain city... it's snowing


In [4]:
df.index

RangeIndex(start=0, stop=1599999, step=1)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599999 entries, 0 to 1599998
Data columns (total 6 columns):
 #   Column                                                                                                               Non-Null Count    Dtype 
---  ------                                                                                                               --------------    ----- 
 0   0                                                                                                                    1599999 non-null  int64 
 1   1467810369                                                                                                           1599999 non-null  int64 
 2   Mon Apr 06 22:19:45 PDT 2009                                                                                         1599999 non-null  object
 3   NO_QUERY                                                                                                             1599999 non-null  object
 4   _

In [6]:
df.shape

(1599999, 6)

In [7]:
# Naming the column names

column_names = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('training.1600000.processed.noemoticon.csv', names = column_names, encoding= 'ISO-8859-1')
df.head(3)

,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...


In [8]:
## Counting Missing Datas
df.isnull().sum()

target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64

In [9]:
## the dsitribution of Target Colummns

df['target'].value_counts()

## this is equally distirbuited data

target
0    800000
4    800000
Name: count, dtype: int64

### Convert target '4' to '1'

In [10]:
df.replace({'target':{4:1}}, inplace=True)

In [11]:
df.shape

(1600000, 6)

In [12]:
df['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

1. 0 ---> negative sentiment
2. 1 ---> postove sentiment


In [15]:
def stemming(content):
    import re
    from nltk.stem.porter import PorterStemmer
    from nltk.corpus import stopwords
    
    # Initialize stemmer and set up stopwords as a fast lookup SET
    port_stem = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    
    # FIX: Added '^' to remove everything EXCEPT letters
    stemmed_content = re.sub('[^a-zA-Z]', ' ', str(content))
    
    # Lowercase
    stemmed_content = stemmed_content.lower()
    
    # Split into words (only once)
    stemmed_content = stemmed_content.split()
    
    # Stem and remove stopwords using the optimized set
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stop_words]

    # Join the words back into a single string (standard for ML inputs)
    return " ".join(stemmed_content)

In [16]:
from pandarallel import pandarallel

# Initialize (uses all available CPU cores)
pandarallel.initialize(progress_bar=True)

# Swap .apply() for .parallel_apply()
df['stemmed_content'] = df['text'].parallel_apply(stemming)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [17]:
df

,target,id,date,flag,user,text,stemmed_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see
...,...,...,...,...,...,...,...
1599995,1,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...,woke school best feel ever
1599996,1,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...,thewdb com cool hear old walt interview http b...
1599997,1,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...,readi mojo makeov ask detail
1599998,1,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...,happi th birthday boo alll time tupac amaru sh...


In [18]:
X = df['stemmed_content']
Y = df = df['target']

In [19]:
X.head(5)

0    switchfoot http twitpic com zl awww bummer sho...
1    upset updat facebook text might cri result sch...
2    kenichan dive mani time ball manag save rest g...
3                      whole bodi feel itchi like fire
4                        nationwideclass behav mad see
Name: stemmed_content, dtype: object

In [20]:
Y.head()

0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

## Splitting the data

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify = Y, random_state = 2)

In [24]:
print(X_train)
print(y_train)

1570269                          watch saw iv drink lil wine
1273074                                         hatermagazin
88479      even though favourit drink think vodka coke wi...
254604                   think hand got burnt sun today hurt
667941     took mazi dr shot today come find ear infect p...
                                 ...                        
941805                                       threewink cheer
1007131    vote livewir play live smith tomorrow night su...
1460311                               eager monday afternoon
929226     hope everyon mother great day wait hear guy st...
526253                      love wake folger bad voic deeper
Name: stemmed_content, Length: 1280000, dtype: object
1570269    1
1273074    1
88479      0
254604     0
667941     0
          ..
941805     1
1007131    1
1460311    1
929226     1
526253     0
Name: target, Length: 1280000, dtype: int64


In [25]:
print(X.shape, X_train.shape, X_test.shape)

(1600000,) (1280000,) (320000,)


## Vectorization

In [26]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

## Training the model


In [28]:
model = LogisticRegression(max_iter=1000, n_jobs = 1)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## Accuracy Score

In [31]:
X_train_pred = model.predict(X_train)
training_data_acc = accuracy_score(y_train, X_train_pred)

print("The accuracy Score of the Training Data is: {}", training_data_acc)

The accuracy Score of the Training Data is: {} 0.79871953125


In [32]:
X_test_pred = model.predict(X_test)
test_data_acc = accuracy_score(y_test, X_test_pred)

print("The accuracy Score of the test Data is: {}", test_data_acc)

The accuracy Score of the test Data is: {} 0.77668125


### Model's accuracy is 77.6%

## Saving the Model

In [33]:
import pickle

In [34]:
filename = 'trained_sentiment_analysis_model.sav'
pickle.dump(model, open(filename, 'wb'))

## Using the saved model for future predictions

In [36]:
loaded_model = pickle.load(open('trained_sentiment_analysis_model.sav', 'rb'))

In [43]:
X_new = X_test[197]
#print(Y_test[197])

prediction = loaded_model.predict(X_new)
print(prediction)

if (prediction[0] == 0):
    print('Negative Tweet')

else:
    print('Positive Tweet')

[0]
Negative Tweet
